In [1]:
import pandas as pd
from astroquery.mast import Catalogs
import numpy as np

In [ ]:
class Gaia_Search():

    def __init__(self, gaia_xmatch):
        self.gaia_xmatch = gaia_xmatch
        self.null = gaia_xmatch.Gaia_ID.isnull()
        self.gaia_info = self.initialise_result_df()
        self.error = []

    def initialise_result_df(self):
        # Test query to set up the result dataframe
        test_id = 'Gaia DR3 5022972468946212352'

        catalog_data = Catalogs.query_object(test_id, catalog="Gaia", radius=0.001)
        df = catalog_data.to_pandas()

        d = df.copy()
        d.insert(0, 'groups_index', 1)
        d.insert(1, 'input_id', 'test')

        return d.iloc[0:0]

    def _candidate_ids(self, gaia_id):
        gaia_id = str(gaia_id).strip()
        if gaia_id == '' or gaia_id.lower() == 'nan':
            return []

        if gaia_id.startswith('Gaia '):
            return [gaia_id]

        if gaia_id.endswith('.0'):
            gaia_id = gaia_id[:-2]

        return [
            f'Gaia DR3 {gaia_id}',
            f'Gaia EDR3 {gaia_id}',
            gaia_id,
        ]

    def query_gaia(self, r=0.01, verbose=True):
        for i, gaia_id in enumerate(self.gaia_xmatch.Gaia_ID):

            # Exceptions
            if self.null.iloc[i]:
                continue
            if pd.isna(gaia_id):
                continue

            candidates = self._candidate_ids(gaia_id)
            if len(candidates) == 0:
                continue

            if verbose:
                print(i, '----{}%'.format(round(i / len(self.gaia_xmatch) * 100, 2)))

            last_error = None
            matched = False

            for object_name in candidates:
                try:
                    catalog_data = Catalogs.query_object(object_name, catalog="Gaia", radius=r)
                    data = catalog_data.to_pandas()

                    if data.empty:
                        continue

                    data = data.iloc[[0]].copy()
                    data.insert(0, 'groups_index', i)
                    data.insert(1, 'input_id', object_name)

                    self.gaia_info = pd.concat([self.gaia_info, data], ignore_index=True)
                    matched = True
                    break
                except Exception as e:
                    last_error = e

            if not matched:
                self.error.append(i)
                if verbose:
                    if last_error is not None:
                        print(f'Query failed for {candidates[0]}: {type(last_error).__name__}: {last_error}')
                    else:
                        print(f'Could not find a match for {candidates[0]}')

        self.gaia_info.to_pickle('gaia-r0_01.pkl')
        np.save('err_r0_01.npy', self.error)